# Model Families: Mistral, Gemma & Qwen

Almost every open LLM is a decoder-only Transformer, yet the families differ in a handful of design choices that decide **memory, speed, context length and multilingual / code quality**. In this notebook we compare three families by inspecting their real Hugging Face configs (no weights are downloaded):

| Family | Developer | Signature ideas |
|---|---|---|
| **Mistral / Mixtral** | Mistral AI | Grouped-Query Attention, sliding-window attention, sparse MoE (Mixtral) |
| **Gemma 2** | Google DeepMind | Alternating local/global attention, logit soft-capping, GeGLU, very large vocabulary, tied embeddings |
| **Qwen 2.5 / Qwen3** | Alibaba | GQA, QKV bias, long-context RoPE scaling, strong multilingual & code data, fine-grained MoE (Qwen3) |

**What you will do**
1. Pull configs and compare architecture hyper-parameters side by side
2. Count exact parameters (dense vs. total vs. *active* for MoE) on the `meta` device
3. Compute the **KV-cache cost** that GQA buys you
4. Visualize full vs. sliding-window vs. alternating attention masks
5. Compare **tokenizer efficiency** (fertility) across families

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import math, torch, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
pd.set_option("display.width", 200)

## 1. Pull the Configs

A config is a few KB of JSON, so we can inspect 7B-30B models instantly. `try/except` keeps the notebook running if a gated repo is not accessible from your account.

In [ ]:
MODELS = {
    "Qwen2.5-0.5B":   "Qwen/Qwen2.5-0.5B",
    "Gemma-2-2B":     "google/gemma-2-2b",
    "Mistral-7B":     "mistralai/Mistral-7B-v0.1",
    "Qwen2.5-7B":     "Qwen/Qwen2.5-7B",
    "Gemma-2-9B":     "google/gemma-2-9b",
    "Mixtral-8x7B":   "mistralai/Mixtral-8x7B-v0.1",
    "Qwen3-30B-A3B":  "Qwen/Qwen3-30B-A3B",
}
configs = {}
for name, repo in MODELS.items():
    try:
        configs[name] = AutoConfig.from_pretrained(repo)
    except Exception as e:
        print(f"skip {name}: {type(e).__name__}")

def g(c, *keys, default=None):
    for k in keys:
        v = getattr(c, k, None)
        if v is not None: return v
    return default

rows = []
for name, c in configs.items():
    h, nh = c.hidden_size, c.num_attention_heads
    rows.append({
        "model": name, "type": c.model_type,
        "layers": c.num_hidden_layers, "d_model": h,
        "q_heads": nh, "kv_heads": g(c, "num_key_value_heads", default=nh),
        "head_dim": g(c, "head_dim", default=h // nh),
        "ffn_dim": g(c, "moe_intermediate_size", "intermediate_size"),
        "vocab": c.vocab_size,
        "ctx": c.max_position_embeddings,
        "window": g(c, "sliding_window", default="-"),
        "act": g(c, "hidden_activation", "hidden_act"),
        "tied_emb": c.tie_word_embeddings,
        "experts": g(c, "num_local_experts", "num_experts", default=1),
        "top_k": g(c, "num_experts_per_tok", default=1),
    })
df = pd.DataFrame(rows).set_index("model")
df

**Things to notice**
- **GQA everywhere**: `kv_heads < q_heads` in all families (Mistral 32→8, Qwen2.5-7B 28→4). Several query heads share one K/V head.
- **Gemma's vocabulary is ~256k** vs. 32k (Mistral) and ~152k (Qwen). Great multilingual coverage, but the embedding matrix becomes a large share of a small model — which is why Gemma ties input/output embeddings.
- **Activation**: Mistral and Qwen use SwiGLU (`silu`); Gemma uses GeGLU (`gelu_pytorch_tanh`).
- **Gemma `head_dim` is decoupled** from `d_model / q_heads` (256 vs. 288): attention width is a free design choice.
- Mixtral and Qwen3-MoE replace the dense FFN with `experts` expert FFNs and route each token to `top_k` of them (see **4.4 Mixture of Experts**).

## 2. Exact Parameter Counts — Total vs. Active

Instantiating a model on PyTorch's **`meta` device** creates the full module graph without allocating memory, so we can count parameters of a 30B model on a laptop.

For MoE models, only `top_k` of `experts` FFNs run per token, so **active parameters** are much lower than **total parameters** — this is what determines per-token compute, while total parameters determine memory.

In [ ]:
def count_params(c):
    with torch.device("meta"):
        m = AutoModelForCausalLM.from_config(c)
    total = sum(p.numel() for p in m.parameters())
    # parameters living inside expert FFN stacks (name contains 'experts')
    expert = sum(p.numel() for n, p in m.named_parameters() if ".experts" in n)
    E, k = g(c, "num_local_experts", "num_experts", default=1), g(c, "num_experts_per_tok", default=1)
    active = total - expert + (expert * k / E if E > 1 else 0)
    emb = c.vocab_size * c.hidden_size
    return total, active, emb

pc = []
for name, c in configs.items():
    t, a, e = count_params(c)
    pc.append({"model": name, "total_B": t/1e9, "active_B": a/1e9, "embedding_share_%": 100*e/t})
pc = pd.DataFrame(pc).set_index("model").round(2)
pc

Gemma-2-2B is a "2B" model whose embedding table alone is a large share of its parameters, while Mixtral and Qwen3-30B-A3B show the MoE trade-off: **memory of a big model, per-token compute of a small one** (the "A3B" in the Qwen3 name means ~3B *active*).

## 3. What GQA Buys You: KV-Cache Memory

During generation every layer caches K and V for all previous tokens:

$$\text{KV bytes / token} = 2 \times L \times n_{kv} \times d_{head} \times \text{bytes per element}$$

With multi-head attention $n_{kv} = n_{q}$; GQA shrinks it by $n_q / n_{kv}$. (MLA from **4.5** compresses it even further.)

In [ ]:
def kv_bytes_per_token(c, bytes_el=2, mha=False):
    nh = c.num_attention_heads
    nkv = nh if mha else g(c, "num_key_value_heads", default=nh)
    hd = g(c, "head_dim", default=c.hidden_size // nh)
    return 2 * c.num_hidden_layers * nkv * hd * bytes_el

ctx = 32_768
kv = pd.DataFrame({
    "KV/token (MHA, KB)": {n: kv_bytes_per_token(c, mha=True)/1024 for n, c in configs.items()},
    "KV/token (actual, KB)": {n: kv_bytes_per_token(c)/1024 for n, c in configs.items()},
})
kv["GQA saving (x)"] = kv.iloc[:, 0] / kv.iloc[:, 1]
kv[f"KV @ {ctx//1024}k ctx (GB, fp16)"] = kv["KV/token (actual, KB)"] * ctx / 1024**2
kv.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(kv)); w = 0.38
ax.bar(x - w/2, kv.iloc[:, 0], w, label="if MHA", color="#c9c9c9")
ax.bar(x + w/2, kv.iloc[:, 1], w, label="actual (GQA)", color="#1f77b4")
ax.set_xticks(x); ax.set_xticklabels(kv.index, rotation=30, ha="right")
ax.set_ylabel("KV cache per token (KB, fp16)"); ax.set_title("KV-cache cost per token"); ax.legend()
plt.tight_layout(); plt.show()

## 4. Attention Patterns: Full vs. Sliding Window vs. Alternating

- **Mistral 7B** uses *sliding-window attention* (window 4096): each token attends only to the last $W$ tokens. Because information propagates layer by layer, the effective receptive field is about $L \times W$, while attention cost and the KV cache are capped at $W$.
- **Gemma 2** *alternates* local (sliding-window) and global (full) layers, keeping cheap local layers while still allowing long-range mixing in the global ones.
- **Qwen2.5** uses full attention and relies on RoPE scaling (YaRN) plus GQA for long contexts.

In [ ]:
def causal(n): return np.tril(np.ones((n, n)))
def sliding(n, w): return np.tril(np.ones((n, n))) - np.tril(np.ones((n, n)), -w)

n, w = 16, 4
fig, axs = plt.subplots(1, 4, figsize=(14, 3.4))
panels = [("Full causal\n(Qwen2.5)", causal(n)),
          (f"Sliding window W={w}\n(Mistral)", sliding(n, w)),
          ("Gemma 2 — even layer\n(local)", sliding(n, w)),
          ("Gemma 2 — odd layer\n(global)", causal(n))]
for ax, (t, m) in zip(axs, panels):
    ax.imshow(m, cmap="Blues", vmin=0, vmax=1.2); ax.set_title(t, fontsize=10)
    ax.set_xlabel("key position"); ax.set_ylabel("query position")
plt.tight_layout(); plt.show()

## 5. Tokenizers: Same Text, Different Token Counts

Tokenizer efficiency (**fertility** = tokens per word/character) changes real cost: more tokens means more compute, a fuller context window, and a higher API bill. A bigger vocabulary generally means fewer tokens for non-English text and code.

In [ ]:
TOK_REPOS = {"Mistral": "mistralai/Mistral-7B-v0.1", "Gemma-2": "google/gemma-2-2b", "Qwen2.5": "Qwen/Qwen2.5-0.5B"}
toks = {}
for n, r in TOK_REPOS.items():
    try: toks[n] = AutoTokenizer.from_pretrained(r)
    except Exception as e: print(f"skip tokenizer {n}: {type(e).__name__}")

samples = {
    "English":  "The quick brown fox jumps over the lazy dog while the model learns to predict the next token.",
    "Python":   "def fib(n):\n    return n if n < 2 else fib(n - 1) + fib(n - 2)\n\nprint([fib(i) for i in range(10)])",
    "Chinese":  "大型语言模型通过预测下一个词来学习语言的统计规律。",
    "Hindi":    "बड़े भाषा मॉडल अगले शब्द की भविष्यवाणी करके भाषा सीखते हैं।",
    "Math":     "Let $f(x)=\\int_0^x e^{-t^2}\\,dt$. Then $f'(x)=e^{-x^2}$.",
}
tok_df = pd.DataFrame({tn: {k: len(t.encode(v, add_special_tokens=False)) for k, v in samples.items()}
                       for tn, t in toks.items()})
tok_df

In [ ]:
(tok_df.div(tok_df.min(axis=1), axis=0)).round(2).plot.bar(figsize=(9, 3.8), rot=0)
plt.ylabel("tokens relative to best tokenizer"); plt.title("Tokenizer fertility by content type")
plt.axhline(1, color="k", lw=0.5); plt.tight_layout(); plt.show()

## 6. Design Choices at a Glance

| Choice | Mistral / Mixtral | Gemma 2 | Qwen2.5 / Qwen3 |
|---|---|---|---|
| Attention | GQA + sliding window (7B v0.1) | GQA + alternating local/global | GQA, full attention |
| FFN activation | SwiGLU | GeGLU | SwiGLU |
| Normalization | Pre-RMSNorm | Pre- **and** post-RMSNorm | Pre-RMSNorm |
| Positional encoding | RoPE | RoPE | RoPE (+ YaRN / long-context scaling) |
| Vocabulary | 32k | 256k | ~152k |
| Embeddings | Untied | Tied | Tied in small models, untied in large |
| Stability tricks | – | Attention & final-logit soft-capping | QKV bias, QK-norm (Qwen3) |
| MoE | Mixtral 8x7B, top-2 of 8 | – | Qwen3-30B-A3B, top-8 of 128 |

## Key Takeaways

1. **The families are variations on one recipe** (RoPE + RMSNorm + gated FFN + GQA); differences are in attention pattern, vocabulary, and how FFN capacity is spent.
2. **GQA is the main lever for serving cost** — it cuts the KV cache by $n_q/n_{kv}$ with little quality loss.
3. **Vocabulary size is a trade-off**: bigger vocab → fewer tokens for multilingual text and code, but a larger embedding/softmax and a bigger share of small models.
4. **Total vs. active parameters**: MoE models are sized by memory (total) but run at the speed of their active parameters.
5. **Pick a family by workload**: Qwen for multilingual/code/long context, Gemma for strong small models, Mistral/Mixtral for efficient inference and a mature ecosystem.